# 4. Catalogues and metadata: STAC, records, checksums: solutions

Exercises of [chapter 4](../../04_catalogs_and_metadata.md). Run the setup cell first.

These are the solutions; try the exercise notebook first.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import hashlib, json
import pystac
import yaml
from data_acquisition.catalog import build, descriptors, stac
from data_acquisition.catalog import summary as summaries
from data_acquisition.readers import eml
from data_acquisition.sources import checksums
from data_acquisition.steps.curate import curate_item

OUT = ROOT / "output/manual/stac/training_pond"


**1. Build a catalogue.** Build the sample dataset with the reference summary, run `curate` on its items, and save it under `output/manual/stac/training_pond`. Walk the tree: catalogue, collections, items.

> **From the chapter.** §1: a catalogue links collections, a collection groups items, and an item points to assets.
>
> **Code.** `summaries.load` in [`data_acquisition/catalog/summary.py`](../../../../data_acquisition/catalog/summary.py), `build.build` in [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py), `curate_item` in [`data_acquisition/steps/curate.py`](../../../../data_acquisition/steps/curate.py), and `stac.save` in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py). The reference summary is [`docs/manual/solutions/training_pond.yaml`](../training_pond.yaml).

In [ ]:
s = summaries.load(str(ROOT / "docs/manual/solutions/training_pond.yaml"))
catalog, report = build.build(SAMPLE, s)
for item in catalog.get_items(recursive=True):
    curate_item(item, s)
path = stac.save(catalog, OUT)
print("saved:", path, "| build report:", {k: v for k, v in report.items() if v} or "empty")
for collection in catalog.get_children():
    print("collection", collection.id)
    for item in collection.get_items():
        print("   item", item.id)

**2. Read an item.** Open the orthomosaic item's JSON. Find its footprint, time, `gsd`, projection fields, assets and their roles, and every `beaver:` property. Which descriptors are still missing?

> **From the chapter.** §1 for the item's layout on disk, §2 for the extensions and the `beaver:` properties.
>
> **Code.** The item's JSON file, and `descriptors.missing` in [`data_acquisition/catalog/descriptors.py`](../../../../data_acquisition/catalog/descriptors.py).

In [ ]:
item_id = "training_pond_2023-07-12_orthomosaic"
d = json.loads((OUT / "training_pond" / item_id / f"{item_id}.json").read_text())
print("footprint:", d["geometry"]["type"], "| bbox:", [round(v, 6) for v in d["bbox"]])
print("time:", d["properties"]["datetime"], "| gsd:", d["properties"]["gsd"])
print("projection:", {k: v for k, v in d["properties"].items() if k.startswith("proj:") and k != "proj:transform"})
print("assets:", {k: v["roles"] for k, v in d["assets"].items()})
for k, v in d["properties"].items():
    if k.startswith("beaver:") and k != "beaver:origin":
        print(f"  {k}: {str(v)[:100]}")
print("missing:", descriptors.missing(pystac.Item.from_dict(d)))

**3. Validate.** Validate the catalogue against the STAC schemas with `stac.validate`.

> **From the chapter.** §1: STAC is a published standard, with schemas a catalogue can be checked against.
>
> **Code.** `stac.load` and `stac.validate`, in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py). The first run needs the network, for the schemas.

In [ ]:
errors = stac.validate(stac.load(OUT / "catalog.json"))
print("valid" if not errors else errors)

**4. Verify a checksum.** Compute the MD5 of the sample orthomosaic; compare it with the record (`metadata.xml`) and with the item's `file:checksum`. Decode the multihash prefix.

> **From the chapter.** §4: a checksum fingerprints a file's bytes, and `file:checksum` stores it as a multihash: a code for the algorithm, the length, then the digest.
>
> **Code.** `eml.read` in [`data_acquisition/readers/eml.py`](../../../../data_acquisition/readers/eml.py) reads the record; Python's `hashlib` computes the MD5.

In [ ]:
ortho = DATASET / "ortho_2023-07-12.tif"
md5 = hashlib.md5(ortho.read_bytes()).hexdigest()
record = eml.read((DATASET / "metadata.xml").read_text())
print("file:   ", md5)
print("record: ", record["entities"]["ortho_2023-07-12.tif"]["checksum"])
multihash = d["assets"]["orthomosaic"]["file:checksum"]
print("item:   ", multihash, "-> code", multihash[:2], "(md5), length", int(multihash[2:4], 16), "bytes")
print("all agree:", md5 == record["entities"]["ortho_2023-07-12.tif"]["checksum"][1] == multihash[4:])

**5. A multipart ETag.** Compute the S3 multipart ETag of the sample orthomosaic for a part size of 5 MiB and of 64 KiB, by hand. Check both with `checksums.check(path, "s3etag", value)`. What does `check` return for an ETag of a part size it does not try?

> **From the chapter.** §4: a multipart ETag is the MD5 of the parts' MD5s, so checking it needs the part size.
>
> **Code.** `checksums.check`, in [`data_acquisition/sources/checksums.py`](../../../../data_acquisition/sources/checksums.py). You write the ETag itself by hand with `hashlib`.

In [ ]:
def s3_etag(path, part):
    data = path.read_bytes()
    digests = [hashlib.md5(data[i:i + part]).digest() for i in range(0, len(data), part)]
    return f"{hashlib.md5(b''.join(digests)).hexdigest()}-{len(digests)}"

for part in (5 * 1024 * 1024, 64 * 1024):
    value = s3_etag(ortho, part)
    print(f"part {part:>9,d} bytes: {value} -> check: {checksums.check(ortho, 's3etag', value)}")
# 5 MiB: one part, checkable (True). 64 KiB: several parts of a size the pipeline does not try: None, undecided,
# and the pipeline falls back to another checksum instead of guessing.

**6. Find it in the code.** Where is the multihash prefix of each algorithm defined, and which function fills `beaver:provenance`?

> **From the chapter.** §4 for the multihash, §2 for `beaver:provenance`.
>
> **Code.** Look in [`data_acquisition/catalog/stac.py`](../../../../data_acquisition/catalog/stac.py) and [`data_acquisition/catalog/build.py`](../../../../data_acquisition/catalog/build.py).

In [ ]:
import inspect
print("stac.MULTIHASH =", stac.MULTIHASH)
print(inspect.getsourcefile(build._provenance), inspect.getsourcelines(build._provenance)[1])
print(inspect.getdoc(build._provenance))